#Early and Drift-Type–Aware model building for Improving PELT-Based Time-Series Forecasting

##Section 1 : Loading Libraries and Data

In [ ]:
!pip install ruptures tensorflow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ruptures as rpt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from xgboost import XGBRegressor

In [ ]:
# Load dataset
file_path = "/content/drive/MyDrive/ele_6mon.xlsx"
df = pd.read_excel(file_path)

# Inspect
print(df.head())
print(df.info())

# Assume timestamp + target column
df.columns = [col.lower() for col in df.columns]

# Rename if needed
# df.rename(columns={'timestamp_col':'timestamp', 'target_col':'value'}, inplace=True)
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y %H:%M')
df = df.sort_values('date').reset_index(drop=True)
df.set_index('date', inplace=True)

print("Data loaded successfully:", df.shape)

                 date  mels_S  lig_S  mels_N     hvac_N     hvac_S
0 2018-01-01 01:00:00     1.2    0.2     7.5  37.400002  19.500000
1 2018-01-01 01:15:00     1.3    0.2     6.8  37.500000  19.889999
2 2018-01-01 01:30:00     1.1    0.2     7.4  38.000000  19.299999
3 2018-01-01 01:45:00     1.2    0.2     7.7  37.200001  18.889999
4 2018-01-01 02:00:00     1.1    0.2     7.3  37.400002  24.700001
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17037 entries, 0 to 17036
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    17037 non-null  datetime64[ns]
 1   mels_S  17030 non-null  float64       
 2   lig_S   17032 non-null  float64       
 3   mels_N  17033 non-null  float64       
 4   hvac_N  16365 non-null  float64       
 5   hvac_S  16365 non-null  float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 798.7 KB
None
Data loaded successfully: (17037, 5)


##Section 2 : Data Preprocessing

In [ ]:
#creating target variable
df['total_energy'] = df[['mels_s', 'lig_s', 'mels_n', 'hvac_n', 'hvac_s']].sum(axis=1)

#round upto 2 decimals
cols_to_round = ['mels_s','lig_s','mels_n','hvac_n','hvac_s','total_energy']
df[cols_to_round] = df[cols_to_round].round(2)

print(df.head())

# Handle missing values
df = df.ffill()

# Remove outliers (optional)
def remove_outliers(series):
    q1, q3 = np.percentile(series, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return np.clip(series, lower, upper)

df['total_energy'] = remove_outliers(df['total_energy'])

# Normalize later after features
print("Preprocessing done")

                     mels_s  lig_s  mels_n  hvac_n  hvac_s  total_energy
date                                                                    
2018-01-01 01:00:00     1.2    0.2     7.5    37.4   19.50         65.80
2018-01-01 01:15:00     1.3    0.2     6.8    37.5   19.89         65.69
2018-01-01 01:30:00     1.1    0.2     7.4    38.0   19.30         66.00
2018-01-01 01:45:00     1.2    0.2     7.7    37.2   18.89         65.19
2018-01-01 02:00:00     1.1    0.2     7.3    37.4   24.70         70.70
Preprocessing done


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17037 entries, 2018-01-01 01:00:00 to 2018-06-30 23:45:00
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mels_s        17037 non-null  float64
 1   lig_s         17037 non-null  float64
 2   mels_n        17037 non-null  float64
 3   hvac_n        17037 non-null  float64
 4   hvac_s        17037 non-null  float64
 5   total_energy  17037 non-null  float64
dtypes: float64(6)
memory usage: 931.7 KB


##Section 3 : Feature Engineering

In [ ]:
def create_features(df):

    df_feat = df.copy()

    # Time features
    df_feat['hour'] = df_feat.index.hour
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['dayofyear'] = df_feat.index.dayofyear # Added this line

    # Cyclic encoding
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour']/24)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour']/24)

    df_feat['dow_sin'] = np.sin(2 * np.pi * df_feat['dayofweek']/7)
    df_feat['dow_cos'] = np.cos(2 * np.pi * df_feat['dayofweek']/7)

    # Lag features
    df_feat['lag_1'] = df_feat['total_energy'].shift(1)
    df_feat['lag_24'] = df_feat['total_energy'].shift(24)
    df_feat['lag_168'] = df_feat['total_energy'].shift(168)

    # Rolling stats
    df_feat['roll_mean_24'] = df_feat['total_energy'].rolling(24).mean()
    df_feat['roll_std_24'] = df_feat['total_energy'].rolling(24).std()

    df_feat = df_feat.dropna()

    return df_feat

df_feat = create_features(df)

print("Features created:", df_feat.shape)

Features created: (16869, 18)


In [ ]:
df_feat.head()

,mels_s,lig_s,mels_n,hvac_n,hvac_s,total_energy,hour,dayofweek,dayofyear,hour_sin,hour_cos,dow_sin,dow_cos,lag_1,lag_24,lag_168,roll_mean_24,roll_std_24
date,,,,,,,,,,,,,,,,,,
2018-01-02 19:00:00,7.70,4.20,21.8,37.2,19.7,90.60,19,1,2,-0.965926,0.258819,0.781831,0.62349,119.39,93.90,65.80,94.735000,18.327215
2018-01-02 19:15:00,8.19,4.09,24.7,60.0,22.5,119.48,19,1,2,-0.965926,0.258819,0.781831,0.62349,90.60,64.40,65.69,97.030000,17.804605
2018-01-02 19:30:00,8.30,4.20,24.0,37.9,30.3,104.70,19,1,2,-0.965926,0.258819,0.781831,0.62349,119.48,93.80,66.00,97.484167,17.857572
2018-01-02 19:45:00,7.80,4.30,23.5,67.0,18.7,121.30,19,1,2,-0.965926,0.258819,0.781831,0.62349,104.70,93.39,65.19,98.647083,18.477379
2018-01-02 20:00:00,7.40,2.00,24.0,37.4,43.1,113.90,20,1,2,-0.866025,0.500000,0.781831,0.62349,121.30,97.40,70.70,99.334583,18.734140


In [ ]:
#columns
df_feat.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16869 entries, 2018-01-02 19:00:00 to 2018-06-30 23:45:00
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mels_s        16869 non-null  float64
 1   lig_s         16869 non-null  float64
 2   mels_n        16869 non-null  float64
 3   hvac_n        16869 non-null  float64
 4   hvac_s        16869 non-null  float64
 5   total_energy  16869 non-null  float64
 6   hour          16869 non-null  int32  
 7   dayofweek     16869 non-null  int32  
 8   dayofyear     16869 non-null  int32  
 9   hour_sin      16869 non-null  float64
 10  hour_cos      16869 non-null  float64
 11  dow_sin       16869 non-null  float64
 12  dow_cos       16869 non-null  float64
 13  lag_1         16869 non-null  float64
 14  lag_24        16869 non-null  float64
 15  lag_168       16869 non-null  float64
 16  roll_mean_24  16869 non-null  float64
 17  roll_std_24   16869 non-null  floa

##Section 4 : Model Training

###Section 4.1 : Splitting of Data

In [ ]:
#splitting of data
split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

In [ ]:
print(f"Total rows  : {len(df_feat)}")
print(f"Train rows  : {len(train_df)}")
print(f"Test  rows  : {len(test_df)}")
print(f"Train period: {train_df.index[0]} → {train_df.index[-1]}")
print(f"Test  period: {test_df.index[0]}  → {test_df.index[-1]}\n")

Total rows  : 17037
Train rows  : 13629
Test  rows  : 3408
Train period: 2018-01-01 01:00:00 → 2018-05-26 11:45:00
Test  period: 2018-05-26 12:00:00  → 2018-06-30 23:45:00



###Section 4.2 : XGBoost Model

In [ ]:
# 2. XGBOOST
print("=" * 50)
print("Training XGBoost ...")
print("=" * 50)

FEATURES = ['hour', 'minute', 'dayofweek', 'dayofyear', 'month', 'week',
            'time_idx', 'lag_1', 'lag_4', 'lag_96', 'rolling_4', 'rolling_96']

X_train, y_train = train_df[FEATURES], train_df['TOTAL_ENERGY']
X_test,  y_test  = test_df[FEATURES],  test_df['TOTAL_ENERGY']

xgb_model = XGBRegressor(
    n_estimators    = 500,
    learning_rate   = 0.05,
    max_depth       = 6,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
    verbosity       = 0
)
xgb_model.fit(
    X_train, y_train,
    eval_set        = [(X_test, y_test)],
    verbose         = False
)

xgb_pred = xgb_model.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)


print(f"XGBoost  MAE : {xgb_mae:.3f}")
print(f"XGBoost  RMSE: {xgb_rmse:.3f}")
print(f"XGBoost  R2: {xgb_r2:.3f}")


Training XGBoost ...


KeyError: "None of [Index(['hour', 'minute', 'dayofweek', 'dayofyear', 'month', 'week', 'time_idx',\n       'lag_1', 'lag_4', 'lag_96', 'rolling_4', 'rolling_96'],\n      dtype='object')] are in the [columns]"

###Section 4.3 Sequence Preparation

In [ ]:
# 3. LSTM & GRU  —  sequence preparation

SEQ_LEN    = 96   # 24 hours of 15-min intervals
BATCH_SIZE = 64
EPOCHS     = 50

# Scale only on TOTAL_ENERGY
scaler = MinMaxScaler()
energy_all   = df['TOTAL_ENERGY'].values.reshape(-1, 1)
energy_scaled = scaler.fit_transform(energy_all)

# Build sequences
def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i - seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_seq, y_seq = make_sequences(energy_scaled, SEQ_LEN)

# Align with split_idx (offset by SEQ_LEN because first SEQ_LEN rows have no sequence)
train_end = split_idx - SEQ_LEN   # samples that belong to train
X_train_seq, y_train_seq = X_seq[:train_end],  y_seq[:train_end]
X_test_seq,  y_test_seq  = X_seq[train_end:],  y_seq[train_end:]

# Reshape → (samples, timesteps, features)
X_train_seq = X_train_seq.reshape(-1, SEQ_LEN, 1)
X_test_seq  = X_test_seq.reshape(-1,  SEQ_LEN, 1)

print(f"Sequence length : {SEQ_LEN} steps")
print(f"Train sequences : {len(X_train_seq)}")
print(f"Test  sequences : {len(X_test_seq)}\n")

# ── Callback helpers ───────────────────────────────────────────────────────────
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0)
    ]


KeyError: 'TOTAL_ENERGY'

###Section 4.4 LSTM Model

In [ ]:
print("=" * 50)
print("Training LSTM ...")
print("=" * 50)

tf.random.set_seed(42)
lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
], name='LSTM_Model')

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data = (X_test_seq, y_test_seq),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = get_callbacks(),
    verbose         = 1
)

lstm_pred_scaled = lstm_model.predict(X_test_seq)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
y_test_lstm = scaler.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

lstm_mae  = mean_absolute_error(y_test_lstm, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm, lstm_pred))
lstm_r2 = r2_score(y_test_lstm, lstm_pred)


print(f"\nLSTM  MAE : {lstm_mae:.3f}")
print(f"LSTM  RMSE: {lstm_rmse:.3f}")
print(f"LSTM  R2: {lstm_r2:.3f}")

###Section 4.5 : GRU Model

In [ ]:
print("=" * 50)
print("Training GRU ...")
print("=" * 50)

tf.random.set_seed(42)
gru_model = Sequential([
    GRU(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    GRU(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
], name='GRU_Model')

gru_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
gru_model.summary()

gru_history = gru_model.fit(
    X_train_seq, y_train_seq,
    validation_data = (X_test_seq, y_test_seq),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = get_callbacks(),
    verbose         = 1
)

gru_pred_scaled = gru_model.predict(X_test_seq)
gru_pred = scaler.inverse_transform(gru_pred_scaled).flatten()
y_test_gru = y_test_lstm  # same ground truth

gru_mae  = mean_absolute_error(y_test_gru, gru_pred)
gru_rmse = np.sqrt(mean_squared_error(y_test_gru, gru_pred))
gru_r2 = r2_score(y_test_gru, gru_pred)

print(f"\nGRU   MAE : {gru_mae:.3f}")
print(f"GRU   RMSE: {gru_rmse:.3f}")
print(f"GRU   R2: {gru_r2:.3f}")

###Section 4.6 : Evaluation metrics

In [ ]:
# 6. RESULTS SUMMARY
print("\n" + "=" * 50)
print(f"{'Model':<12} {'MAE':>8} {'RMSE':>8} {'MAPE':>8}")
print("-" * 50)
print(f"{'XGBoost':<12} {xgb_mae:>8.3f} {xgb_rmse:>8.3f} {xgb_r2:>7.3f}")
print(f"{'LSTM':<12} {lstm_mae:>8.3f} {lstm_rmse:>8.3f} {lstm_r2:>7.3f}")
print(f"{'GRU':<12} {gru_mae:>8.3f} {gru_rmse:>8.3f} {gru_r2:>7.3f}")
print("=" * 50)

##Section 5 : Drift Detection

In [ ]:
y_smooth = pd.Series(y).rolling(window=20, center=True).mean()
y_smooth = y_smooth.bfill().ffill().values


def cost_segment(y, start, end):
    segment = y[start:end]
    if len(segment) == 0:
        return 0
    mean = np.mean(segment)
    return np.sum((segment - mean) ** 2)


def pelt(y, penalty):
    n = len(y)

    F = np.zeros(n + 1)        # Optimal cost
    F[0] = -penalty

    cp = {0: []}              # Change points
    R = [0]                   # Candidate set

    for t in range(1, n + 1):
        costs = []

        for tau in R:
            cost = F[tau] + cost_segment(y, tau, t) + penalty
            costs.append((cost, tau))

        F[t], tau_star = min(costs)
        cp[t] = cp[tau_star] + [tau_star]

        # Pruning step
        R_new = []
        for tau in R:
            if F[tau] + cost_segment(y, tau, t) + penalty <= F[t]:
                R_new.append(tau)

        R = R_new + [t]

    return cp[n][1:-1]  # remove initial 0


penalty = 3000
change_points = pelt(y_smooth, penalty)

print("Detected Change Points:", change_points)



In [ ]:
rpt.display(y_smooth, change_points)
plt.title("Data Drift Detection (PELT)")
plt.show()

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(df['date'], y_smooth, label="Smoothed Energy")

for cp in change_points:
    plt.axvline(df['date'].iloc[cp], linestyle='--', color ='red')

plt.title("PELT Change Point Detection (Energy Data)")
plt.xlabel("Time")
plt.ylabel("Energy")
plt.legend()
plt.show()

##Section 6 : Segmentation based on change points

In [ ]:
len(change_points)

In [ ]:
drift_seg = []
i = 0
while i < len(change_points):
  drift_seg.append(y_smooth[change_points[i]:change_points[i+1]])
  i+=2

In [ ]:
import matplotlib.dates as mdates

for k in range(0, len(change_points), 2):
  start_cp = change_points[k]
  end_cp = change_points[k+1]

  plt.figure(figsize=(5, 5))
  plt.plot(df['date'].iloc[start_cp:end_cp], y_smooth[start_cp:end_cp])
  plt.xlabel("Time")
  plt.ylabel("Energy")
  plt.title(f"Energy Data - Segment from {start_cp} to {end_cp}")

  # Explicitly set date format to show year
  formatter = mdates.DateFormatter('%Y-%m-%d %H:%M') # Example: 2018-01-01 10:00
  plt.gca().xaxis.set_major_formatter(formatter)
  plt.gcf().autofmt_xdate() # Auto-format for better readability

  plt.show()

In [ ]:
segment = []
start = 0  # Initialize start
for cp in change_points:
  segment.append(y_smooth[start:cp])
  start = cp
  if cp == change_points[-1]:
    segment.append(y_smooth[start:])

In [ ]:
len(segment)

##Section 7: Drift Classification

In [ ]:
def _slope(arr):
    x = np.arange(len(arr))
    return float(np.polyfit(x, arr, 1)[0])


def classify_drift(prev_seg: pd.Series, curr_seg: pd.Series,
                   train_std: float,
                   abrupt_thresh=1.0, gradual_thresh=0.5, recurrent_thresh=0.75):
    mean_diff  = abs(prev_seg.mean() - curr_seg.mean()) / (train_std + 1e-9)
    sp, sc     = _slope(prev_seg.values), _slope(curr_seg.values)
    slope_diff = abs(sc - sp) / (abs(sp) + 1e-9)

    n_c = min(100, len(prev_seg), len(curr_seg))
    corr = 0.0
    if n_c > 2:
        c = np.corrcoef(prev_seg.values[-n_c:], curr_seg.values[:n_c])[0, 1]
        corr = float(c) if not np.isnan(c) else 0.0

    scores = {"mean_diff_norm": round(mean_diff, 4),
              "slope_diff": round(slope_diff, 4),
              "head_tail_corr": round(corr, 4)}

    if mean_diff > abrupt_thresh:
        return "abrupt"
    elif slope_diff > gradual_thresh:
        return "gradual"
    elif corr > recurrent_thresh:
        return "recurrent"
    else:
        return "stable"


train_std = y_smooth.std()
drift_labels = []
for i in range(1, len(segment)):
    drift_labels.append(classify_drift(pd.Series(segment[i-1]), pd.Series(segment[i]), train_std))

print("Drift labels:", drift_labels)

In [ ]:
print("Drift classifications at each change point:")
for i in range(len(change_points)):
    print(f"Change Point at index {change_points[i]}: {drift_labels[i]}")

##Section 8 : Model Retraining

###Sequnce generation

In [ ]:
change_points = [6452, 7128, 13854, 13957, 15192, 15599]

def split_segments(df, change_points):

    segments = []
    start = 0

    for cp in change_points:
        segments.append(df.iloc[start:cp])
        start = cp

    segments.append(df.iloc[start:])

    return segments

segments = split_segments(df_feat, change_points)

In [ ]:
seq_len = 24
def make_sequences(X, y, seq_len):

    Xs, ys = [], []

    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])

    return np.array(Xs), np.array(ys)

In [ ]:
scaler = MinMaxScaler()
target =  'total_energy'
X_train_scaled = scaler.fit_transform(
    train_df.drop(columns=[target])
)

X_test_scaled = scaler.transform(
    test_df.drop(columns=[target])
)

y_train = train_df[target].values
y_test = test_df[target].values

In [ ]:
X_train_seq, y_train_seq = make_sequences(
    X_train_scaled,
    y_train,
    seq_len
)

X_test_seq, y_test_seq = make_sequences(
    X_test_scaled,
    y_test,
    seq_len
)

###Section 8.1 : XGBOOST Model

In [ ]:
def retrain_xgb_on_drift(
    drift_df,
    memory_df,
    scaler,
    target,
    seq_len
):

    combined_df = pd.concat([
        memory_df.tail(500),
        drift_df
    ])

    X_combined = scaler.transform(
        combined_df.drop(columns=[target])
    )

    y_combined = combined_df[target].values

    X_seq, y_seq = make_sequences(
        X_combined,
        y_combined,
        seq_len
    )

    X_seq_flat = X_seq.reshape(X_seq.shape[0], -1)

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    )

    model.fit(X_seq_flat, y_seq)

    return model

###Section 8.2 : LSTM Model

In [ ]:
def retrain_lstm_on_drift(
    model,
    drift_df,
    memory_df,
    scaler,
    target,
    seq_len
):

    X_drift = scaler.transform(
        drift_df.drop(columns=[target])
    )

    y_drift = drift_df[target].values

    X_drift_seq, y_drift_seq = make_sequences(
        X_drift,
        y_drift,
        seq_len
    )

    X_mem = scaler.transform(
        memory_df.drop(columns=[target])
    )

    y_mem = memory_df[target].values

    X_mem_seq, y_mem_seq = make_sequences(
        X_mem,
        y_mem,
        seq_len
    )

    replay_size = int(len(X_mem_seq) * 0.3)

    idx = np.random.choice(
        len(X_mem_seq),
        replay_size,
        replace=False
    )

    X_old = X_mem_seq[idx]
    y_old = y_mem_seq[idx]

    X_train = np.concatenate([X_old, X_drift_seq])
    y_train = np.concatenate([y_old, y_drift_seq])

    idx = np.random.permutation(len(X_train))

    X_train = X_train[idx]
    y_train = y_train[idx]

    model.optimizer.learning_rate.assign(1e-4)

    early_stop = EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    return model

In [ ]:
history_df = segments[0]

for i in range(1, len(segments)):

    drift_df = segments[i]

    print(f'LSTM Retraining Segment {i}')

    lstm_model = retrain_lstm_on_drift(
        model=lstm_model,
        drift_df=drift_df,
        memory_df=history_df,
        scaler=scaler,
        target=target,
        seq_len=seq_len
    )

    history_df = pd.concat([
        history_df,
        drift_df
    ])


###Section 8.3 : GRU Model

In [ ]:
def retrain_on_drift(
    model,
    drift_df,
    memory_df,
    scaler,
    target,
    seq_len
):

    # Drift Data
    X_drift = scaler.transform(
        drift_df.drop(columns=[target])
    )

    y_drift = drift_df[target].values

    X_drift_seq, y_drift_seq = make_sequences(
        X_drift,
        y_drift,
        seq_len
    )

    # Memory Data
    X_mem = scaler.transform(
        memory_df.drop(columns=[target])
    )

    y_mem = memory_df[target].values

    X_mem_seq, y_mem_seq = make_sequences(
        X_mem,
        y_mem,
        seq_len
    )

    # Replay Memory
    replay_size = int(len(X_mem_seq) * 0.3)

    idx = np.random.choice(
        len(X_mem_seq),
        replay_size,
        replace=False
    )

    X_old = X_mem_seq[idx]
    y_old = y_mem_seq[idx]

    # Combine
    X_train = np.concatenate([X_old, X_drift_seq])
    y_train = np.concatenate([y_old, y_drift_seq])

    # Shuffle
    idx = np.random.permutation(len(X_train))

    X_train = X_train[idx]
    y_train = y_train[idx]

    # Small LR
    model.optimizer.learning_rate.assign(1e-4)

    early_stop = EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    return model

In [ ]:
history_df = segments[0]

for i in range(1, len(segments)):

    drift_df = segments[i]

    print(f'Retraining Segment {i}')

    gru_model = retrain_on_drift(
        model=gru_model,
        drift_df=drift_df,
        memory_df=history_df,
        scaler=scaler,
        target=target,
        seq_len=seq_len
    )

    history_df = pd.concat([
        history_df,
        drift_df
    ])

##Section 9 : Evaluation Metric

###XGBoost

In [ ]:
after_pred_xgb = xgb_model.predict(X_test_xgb)

xgb_after_mae = mean_absolute_error(y_test_seq, after_pred_xgb)
xgb_after_rmse = np.sqrt(mean_squared_error(y_test_seq, after_pred_xgb))
xgb_after_r2 = r2_score(y_test_seq, after_pred_xgb)

print('XGBoost After Retraining')
print('MAE:', xgb_after_mae)
print('RMSE:', xgb_after_rmse)
print('R2:', xgb_after_r2)

###LSTM

In [ ]:
after_pred_lstm = lstm_model.predict(X_test_seq).flatten()

lstm_after_mae = mean_absolute_error(y_test_seq, after_pred_lstm)
lstm_after_rmse = np.sqrt(mean_squared_error(y_test_seq, after_pred_lstm))
lstm_after_r2 = r2_score(y_test_seq, after_pred_lstm)

print('LSTM After Retraining')
print('MAE:', lstm_after_mae)
print('RMSE:', lstm_after_rmse)
print('R2:', lstm_after_r2)

###GRU

In [ ]:
after_pred = gru_model.predict(X_test_seq).flatten()

after_mae = mean_absolute_error(y_test_seq, after_pred)
after_rmse = np.sqrt(mean_squared_error(y_test_seq, after_pred))
after_r2 = r2_score(y_test_seq, after_pred)

print('After Retraining')
print('MAE:', after_mae)
print('RMSE:', after_rmse)
print('R2:', after_r2)